In [1]:
import os
import getpass
from dotenv import load_dotenv
from langchain_core.globals import set_verbose

load_dotenv(override=True)
set_verbose(True)
openrouter_model = os.getenv("MODEL", "openai/gpt-6-luna")

In [2]:
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

# ⛓️ What are Chains in LangChain?

**In one sentence: A chain is an end-to-end wrapper around multiple individual components which are executed in a defined order.**

**Quick Guide to Chains in LangChain:**

- Chains link multiple processes in a set sequence to create complex applications.
- They're useful for:
  - Dividing complicated tasks into simpler steps.
  - Passing information from one step to the next.
  - Adding custom processing or checks between steps.
  - Simplifying the debugging of multi-step operations.

**Basic Chain Building Blocks:**

- **Prompt, model, parser**: the simplest chain, written `prompt | llm | StrOutputParser()`.
- **Sequential chains**: one step's output becomes the next step's input.
- **Routing**: a step decides which chain runs next.
- **Transformation**: a plain Python function that reshapes data between steps.

Chains integrate various components into a cohesive flow, enhancing the capabilities and flexibility of language model applications.


# 🗣️ Your First Chain (pipe syntax)


In [3]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import PromptTemplate

llm = ChatOpenRouter(model=openrouter_model)

prompt = PromptTemplate(
    input_variables=["topic"],
    template = "Write a comedic, parody rap about the following topic: {topic}"
)

# using LCEL
chain = prompt | llm

chain.invoke({"topic": "wasabi flavoured popcorn"})

AIMessage(content='**“Wasabi Popcorn”**\n\n**[Verse 1]**  \nFriday night, movie’s on, I’m ready to snack,  \nGrab a bowl of popcorn—then my sinuses attack.  \nGreen dust on the kernels, looks harmless and neat,  \nOne bite later, I can see through my feet.\n\nMy nose says, “We’re leaving!” My eyes start to rain,  \nI’m watching the trailers through a waterfall pane.  \nThe hero says, “Run!” and I’m already gone—  \nJust me and my nostrils, sprinting through the dawn.\n\n**[Chorus]**  \nWasabi popcorn, crunch with a kick,  \nTiny green thunderbolts—fast, loud, and quick!  \nSweet movie moment? Not on this block:  \nThe plot’s gone missing—I’m negotiating with a sock.\n\n**[Verse 2]**  \nI offered one kernel to my friend named Lee,  \nLee said, “That’s not popcorn—that’s a dare from a tree!”  \nHe puffed out his cheeks, tried to act like a champ,  \nThen made the same face as a surprised little lamp.\n\nThe bowl’s on the table, I’m keeping my cool,  \nBut every new handful rewrites the w

In [4]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

chain.invoke({"topic": "wasabi flavoured popcorn"})

'**“Green Heat, Big Pop”**\n\n*Intro*  \nYo, movie night, lights low, snack bowl in the spot—  \nWait, why’s the popcorn glowing? It’s wasabi-flavored. What?\n\n*Verse 1*  \nI grabbed one kernel, said, “This should be a breeze,”  \nNow my nose is doing cartwheels and my eyes are on freeze.  \nIt’s crunchy, it’s salty, it’s got a greenish disguise,  \nTastes like a tiny dragon just sneezed in my fries.\n\nI’m chillin’ on the couch, looking cool, feeling neat—  \nThen the wasabi rolls in with its tap-dancing heat.  \nMy mouth says, “More!” and my sinuses say, “No!”  \nMy eyebrows just moonwalked clean off my forehead, yo.\n\n*Chorus*  \nPop-pop, crunch-crunch, green fire in the bowl,  \nWasabi popcorn got a spicy little soul.  \nPop-pop, snort-snort, tissues on standby—  \nOne more handful and I’m clearing out the sky!\n\n*Verse 2*  \nThe movie’s got suspense, but I’m missing the plot,  \nI’m negotiating with a kernel like, “Please be less hot.”  \nMy friend says, “Try some!”—I say, “Tha

In [5]:
print(chain.invoke({"topic": "wasabi flavoured popcorn"}))

**“Wasabi Popcorn”**

*Yo, pass the bowl—wait, read the label!*  
*This snack’s got a kick and it’s legally unstable!*

I popped one kernel, said, “That’s pretty tame,”  
Then my nose hit orbit and forgot my name.  
Green dust flying like a tiny food storm,  
My taste buds filed a complaint in formal form.

**Chorus:**  
Wasabi popcorn, spicy little cloud,  
Makes you cough politely, then sneeze out loud.  
Crunch goes *pop*, then your sinuses sing—  
Movie night’s now a martial-arts thing!

I’m watching a comedy, tears in my eyes,  
Not from the plot—there’s a dragon inside.  
My friend says, “Try one!” I say, “Sure, why not?”  
Now I’m bargaining with the milk in the pot.

The kernels look innocent, round and polite,  
Then they karate-chop your tongue on sight.  
I reached for a handful, got bold with the scoop—  
Now I’m doing deep breaths in the snack aisle loop.

**Chorus:**  
Wasabi popcorn, spicy little cloud,  
Makes you cough politely, then sneeze out loud.  
Crunch goes *pop

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are PaRappa the Rapper. You spit hot fire flows like lava."),
        ("human", "Write a comedic, parody rap about the following topic: {topic}"),
    ]
)

runnable = prompt | llm | StrOutputParser()

In [7]:
print(runnable.invoke({"topic": "wasabi flavoured popcorn"}))

**“Green Nose, Big Crunch”**

Yo, popped in the kitchen, thought I’d found a snack,  
One handful later, my sinuses packed.  
Wasabi on the popcorn, green dust on my sleeve—  
My nose did a backflip, my eyes took leave!

**Crunch-crunch!** Hot stuff, call the fire brigade,  
My tongue’s doing jazz in a spicy parade.  
I wanted movie munchies, a cozy little treat—  
Now I’m seeing subtitles when I breathe through my feet!

It’s popcorn with punch, it’s a kernel karate,  
A tiny green dragon yelling, “Come and try me!”  
I reach for the bowl—then I pause, reconsider:  
That snack’s got a passport and a one-way trip to my sinuses.

So pass me the water! No, wait—it’s too late!  
I’m sweating like a sauna in a movie-theater seat.  
Wasabi popcorn, you’re weird, you’re intense—  
You make regular butter taste like common sense!


In [8]:
# stream() — instead of waiting for the entire LLM response, 
# it returns chunks of the response as they arrive (token-by-token or in small pieces).

for chunk in runnable.stream({"topic": "wasabi flavoured popcorn"}):
    print(chunk, end="", flush=True)

# chunk — the text piece received
# end="" — don't add a newline after each chunk (keeps text flowing continuously)
# flush=True — force the output to display immediately instead of buffering (so you see text appear in real-time, not all at once at the end)

**

“

Green

 Dust

,

...

 coffee

 in

 the

 morning

!

# 🔄 **Understanding Routing in LangChain**

### **Routing Concept**
- **Purpose**: Adds structure to interactions with LLMs by guiding the flow based on previous step outcomes.

- **Essence**: Determines the next step in a chain dynamically, based on the result of the previous step.

### **Method for Routing**
1. 🔀 **RunnableBranch Usage**:
   - Manages decision-making for the next step in a chain.

### **Example Application**
- **Two-Step Sequence**:

  - **Step 1**: Classifies a question into categories (literature, history, biology, philosophy, or general).

  - **Step 2**: Routes the classified question to a corresponding prompt chain tailored for the identified category.

🎯 **Application Goal**: Showcase RunnableBranch routing in a practical scenario to enhance interactions with language models.

In [9]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

literature_template = """You are a seasoned literature professor with decades of experience analyzing literary works. \
You have a knack for understanding complex narratives and characters and can provide insights into underlying themes and motifs.

Here is a passage or question about a literary work:
{input}
"""

literature_prompt = PromptTemplate.from_template(literature_template)


history_template = """You are a historian with extensive knowledge about world history. \
From ancient civilizations to modern times, you can provide context, insights, and explanations about historical events and figures.

Here is a question about history:
{input}
"""

history_prompt = PromptTemplate.from_template(history_template)

biology_template = """You are a biologist with a passion for understanding the intricacies of life. \
From cellular processes to ecosystem dynamics, you can elucidate biological phenomena with clarity.

Here is a question about biology:
{input}
"""

biology_prompt = PromptTemplate.from_template(biology_template)

philosophy_template = """You are a philosopher who has studied the great thinkers of the past and present. \
You enjoy discussing ethical dilemmas, existential questions, and the nature of reality.

Here is a philosophical query:
{input}
"""

philosophy_prompt = PromptTemplate.from_template(philosophy_template)

general_prompt = PromptTemplate.from_template(
    "You are a helpful assistant. Answer the question as accurately as you can.\n\n{input}"
)


# 🌿 **RunnableBranch Mechanics**

1. **Structure**: A list of (condition, runnable) pairs, plus a default runnable.

2. **Operation**
   - On invocation, sequentially evaluates each condition with the given input.

   - Executes the first runnable where its condition is True.
   
   - If no condition matches, the default runnable is executed.

🔍 **Purpose**: Ensures structured decision-making in chain execution, directing the flow based on specific conditions.

In [10]:
prompt_branch = RunnableBranch(
    (lambda x: x["topic"] == "literature", literature_prompt),
    (lambda x: x["topic"] == "history", history_prompt),
    (lambda x: x["topic"] == "biology", biology_prompt),
    (lambda x: x["topic"] == "philosophy", philosophy_prompt),
    general_prompt,
)

# Input: {"input": "Tell me about Shakespeare", "topic": "literature"}
#      ↓
# Check: topic == "literature"? YES ✓
#      ↓
# Use: literature_prompt (specialized for literature)


# def is_literature(x):
#     return x["topic"] == "literature"

# def is_history(x):
#     return x["topic"] == "history"

# def is_biology(x):
#     return x["topic"] == "biology"

# def is_philosophy(x):
#     return x["topic"] == "philosophy"

# prompt_branch = RunnableBranch(
#     (is_literature, literature_prompt),
#     (is_history, history_prompt),
#     (is_biology, biology_prompt),
#     (is_philosophy, philosophy_prompt),
#     general_prompt,
# )

In [11]:
from typing import Literal
from pydantic import BaseModel

class TopicClassifier(BaseModel):
    "Classify the topic of the user question"

    topic: Literal["literature", "history", "biology", "philosophy", "general"]
    "The topic of the user question. One of 'literature', 'history', 'biology', 'philosophy', or 'general'."


llm = ChatOpenRouter(model=openrouter_model)

classifier_llm = llm.with_structured_output(TopicClassifier)

# Extract the plain topic string so the router's conditions can compare it
classifier_chain = classifier_llm | (lambda c: c.topic)

In [12]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

final_chain = (
    RunnablePassthrough.assign(topic=itemgetter("input") | classifier_chain)
    | prompt_branch
    | llm
    | StrOutputParser()
)

In [13]:
final_chain.invoke({"input": "Describe the Stoic philosophy for a good life."})

'For the Stoics, a good life is a life of **virtue**: using reason well, acting justly, and meeting events with courage and self-command. Their aim was *eudaimonia*—a deep, flourishing way of living—not simply pleasure or success.\n\nA central Stoic distinction is between what is **up to us** and what is not. Our judgments, intentions, and choices are substantially ours to shape; other people’s opinions, the past, and many outcomes are not. Stoicism asks us to put our effort into the former and meet the latter without letting them dictate our character. This does not mean passivity: we should work to improve circumstances, while recognizing that we cannot guarantee the result.\n\nThe Stoics held that virtue—often described through wisdom, justice, courage, and temperance—is the true good. Wealth, health, reputation, and comfort can matter, and may reasonably be preferred, but they do not by themselves make someone good or happy. What matters most is how we pursue or respond to them.\n\

In [14]:
for chunk in final_chain.stream({"input": "Describe the Stoic philosophy for a good life."}):
    print(chunk, end="", flush=True)

For

 the

 Sto

ics

,

...

 well

 whatever

 fortune

 brings

.

# 🧬 Sequential Chains

**Basics of Sequential Chains:**

- Sequential chains are for when you need one language model's output to become another's input.
- They're like an assembly line, where each step's result is the starting point for the next.


In [15]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the language model
llm = ChatOpenRouter(model=openrouter_model)

# Template for the initial rap
template_one = """
You are an Irish rapper, like Jafaris or Kojaque.

Given a topic, it is your job to spit bars of hard-hitting, gritty, dope rap.

Topic: {topic}

"""

# Template for the diss track
template_two = """
You are an extremely competitive Irish Rapper.

Given the rap from another rapper, it's your job to write a diss track which
tears apart the rap and shames the original rapper.

Rap:
{rap}
"""

# Create prompt templates from the defined templates
prompt_template_one = PromptTemplate.from_template(template_one)
prompt_template_two = PromptTemplate.from_template(template_two)

# Define the operation chain
chain = (
    {"rap": prompt_template_one | llm | StrOutputParser()}
    | prompt_template_two
    | llm
    | StrOutputParser()
)


In [16]:
chain.invoke({"topic": "Rain as a metaphor for adversity"})

'**“Forecast: Fraud”**\n\nYou wrote sixteen bars and the rain got top billing,  \nClouds had more character—your verse stayed chilling.  \n“Every drop had a debt”? Who’s collecting that tab?  \nYour metaphors need shelter from the weather you blag.\n\nBuilt a roof from the road? That’s a shaky design;  \nOne gust off the Liffey and it’s flat by nine.  \nYou say storms know your address—well, they’ve come to complain:  \n“Can we get one fresh image? We’re sick of this rain.”\n\nPuddles hold the sky? That’s a shallow reflection;  \nYou looked down, saw yourself, called it introspection.  \nMud on your shoes, doubt in your mouth—  \nThat’s not deep, that’s just walking about.\n\nTin-shed kick drum, roots in the street—  \nYou rhyme like a weather app stuck on repeat.  \n“Grew through the rain”—grand, but let’s make it plain:  \nThe only thing growing here is the mould in your refrain.\n\nI’ve got a storm of my own, and it comes with a warning:  \nYour whole big comeback got washed out by 

In [17]:
for chunk in chain.stream({"topic": "Rain as a metaphor for adversity"}):
    print(chunk, end="", flush=True)

**

“

Forecast

:

 Finished

...

 came

 up

 over

 you

.

# 🔄 **Transformation in Component Chains**

🔧 **Role of Transformation**:
   - Adjusts inputs as they transition between different components.

🌟 **Example Scenario**:
   - **Task**: Handle a lengthy text.
   - **Transformation**: Pick out a slice of the text before passing it on. The `lambda` keeps `split("\n")[921:1021]`, which is lines 922 to 1021 when counted from 1.
   - **Source**: *Meditations* by Marcus Aurelius (Project Gutenberg eBook #2680).
   - **Next Steps**: Pass the shortened text through a series of steps for summarization.

🎯 **Goal**: Demonstrates how transformations can effectively tailor inputs for specific processing needs in a component chain.

In [18]:
# !wget https://www.gutenberg.org/files/2680/2680-0.txt

with open("../../data/2680-0.txt") as f:
    meditations = f.read()

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""Summarize this text: {output_text} Summary:""")

runnable = (
    {"output_text": lambda text: "\n".join(text.split("\n")[921:1021])}
    | prompt
    | llm
    | StrOutputParser()
)
runnable.invoke(meditations)

'The passage urges a life guided by reason, justice, sincerity, and natural affection. Focus on each task as if it were your last, stop chasing other people’s approval, and direct your actions toward a clear purpose. Understand yourself as part of the universe, and accept events beyond your control without treating wealth, status, pleasure, hardship, or even death as inherently good or bad. These are natural occurrences; virtue and vice are what truly matter and are largely within your power. The text also warns that wrongdoing driven by pleasure may reveal greater weakness than wrongdoing driven by anger, and reminds the reader that all worldly things—and their reputations—are fleeting.'

In [20]:
rephrase = PromptTemplate.from_template("""Rephrase this text: {output_text}
In the style of a 90s gangster rapper passionately speaking to his homies.
Rephrased:""")

runnable = (
    {"output_text": lambda text: "\n".join(text.split("\n")[921:1021])}
    | rephrase
    | llm
    | StrOutputParser()
)

runnable.invoke(meditations)

'**II.** Keep it real, my soul: as a Roman, as a man, stay locked in on whatever’s in front of you. Do it with honest weight, natural care, a free mind, and justice. Drop the extra noise—the worries and wild thoughts that crowd your head. You can shake them loose if you treat every move like it might be your last: no ego, no losing your temper, no turning your back on reason, no fake act, no self-love running the show, and no beef with what fate—or God—has put in your path. Look around: the things you need to live right, keep your life on track, and walk a divine road aren’t many. The gods aren’t asking for more than this.\n\n**III.** Get up, soul. Stop playing yourself and running yourself down. Your time to show yourself respect is almost up. A person’s happiness comes from within—but look at you, letting your days run out while you hand your happiness over to other people’s opinions and fantasies.\n\n**IV.** Why let outside events throw you off your game? Take a breath. Learn someth